In [1]:
import struct, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import zipfile, os

# Extract shapefile
os.makedirs('/Users/finn/Desktop/TFG/shp', exist_ok=True)
with zipfile.ZipFile('/Users/finn/Desktop/TFG/lineas_limite.zip', 'r') as z:
    z.extractall('/Users/finn/Desktop/TFG/shp')

def read_dbf(path):
    with open(path, 'rb') as f:
        header = f.read(32)
        num_records = struct.unpack('<I', header[4:8])[0]
        header_size = struct.unpack('<H', header[8:10])[0]
        record_size = struct.unpack('<H', header[10:12])[0]
        fields = []
        while True:
            fd = f.read(32)
            if fd[0] == 0x0D: break
            name = fd[:11].replace(b'\x00',b'').decode('latin-1')
            fields.append((name, fd[11], fd[16]))
        f.seek(header_size)
        rows = []
        for _ in range(num_records):
            rec = f.read(record_size)
            if not rec: break
            row = {}; offset = 1
            for fname, ftype, flen in fields:
                val = rec[offset:offset+flen].decode('latin-1', errors='replace').strip()
                row[fname] = val
                offset += flen
            rows.append(row)
    return pd.DataFrame(rows)

def read_shp_polygons(path):
    polys = []
    with open(path, 'rb') as f:
        f.seek(100)
        while True:
            rec_header = f.read(8)
            if len(rec_header) < 8: break
            content_len = struct.unpack('>I', rec_header[4:8])[0] * 2
            content = f.read(content_len)
            if len(content) < 4: break
            shape_type = struct.unpack('<I', content[:4])[0]
            if shape_type not in (5, 15, 25):
                polys.append([]); continue
            num_parts = struct.unpack('<I', content[36:40])[0]
            num_points = struct.unpack('<I', content[40:44])[0]
            parts = [struct.unpack('<I', content[44+i*4:48+i*4])[0] for i in range(num_parts)]
            pts_offset = 44 + num_parts * 4
            points = []
            for i in range(num_points):
                x, y = struct.unpack('<dd', content[pts_offset+i*16:pts_offset+i*16+16])
                points.append((x, y))
            rings = []
            for i, start in enumerate(parts):
                end = parts[i+1] if i+1 < len(parts) else num_points
                rings.append(points[start:end])
            polys.append(rings)
    return polys

print("Reading shapefiles...")
pen_polys = read_shp_polygons('/Users/finn/Desktop/TFG/shp/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.shp')
can_polys = read_shp_polygons('/Users/finn/Desktop/TFG/shp/SHP_REGCAN95/recintos_municipales_inspire_canarias_regcan95/recintos_municipales_inspire_canarias_regcan95.shp')
pen_dbf = read_dbf('/Users/finn/Desktop/TFG/shp/SHP_ETRS89/recintos_municipales_inspire_peninbal_etrs89/recintos_municipales_inspire_peninbal_etrs89.dbf')
can_dbf = read_dbf('/Users/finn/Desktop/TFG/shp/SHP_REGCAN95/recintos_municipales_inspire_canarias_regcan95/recintos_municipales_inspire_canarias_regcan95.dbf')
pen_dbf['cod_ine'] = pen_dbf['NATCODE'].str[-5:]
can_dbf['cod_ine'] = can_dbf['NATCODE'].str[-5:]

gini_raw = pd.read_csv('/Users/finn/Desktop/TFG/GINI_file.csv', sep='\t', encoding='utf-8-sig')
gini_df = gini_raw[
    gini_raw['Municipalities'].str.match(r'^\d{5} ', na=False) &
    gini_raw['Districts'].isna() & gini_raw['Sections'].isna() &
    (gini_raw['Average income indicators']=='Gini Index') & (gini_raw['Periodo']==2022)
].copy()
gini_df['gini'] = pd.to_numeric(gini_df['Total'].astype(str).str.replace(',','.'), errors='coerce')
gini_df['cod_ine'] = gini_df['Municipalities'].str[:5]
gini_df = gini_df[['cod_ine','gini']].dropna()

pen_gini = pen_dbf[['cod_ine']].merge(gini_df, on='cod_ine', how='left')['gini'].values
can_gini = can_dbf[['cod_ine']].merge(gini_df, on='cod_ine', how='left')['gini'].values

all_gini = np.concatenate([pen_gini, can_gini])
vmin = np.nanpercentile(all_gini, 5)
vmax = np.nanpercentile(all_gini, 95)
cmap = plt.cm.RdYlGn_r
norm = Normalize(vmin=vmin, vmax=vmax)

def gini_to_color(g):
    if np.isnan(g): return (0.82, 0.82, 0.82, 1.0)
    return cmap(norm(g))

def draw_polys(ax, polys, gini_vals):
    from matplotlib.patches import Polygon as MplPolygon
    from matplotlib.collections import PatchCollection
    patches, colors = [], []
    for rings, g in zip(polys, gini_vals):
        if not rings: continue
        outer = rings[0]
        if len(outer) < 3: continue
        pts = np.array(outer)
        patch = MplPolygon(pts, closed=True)
        patches.append(patch)
        colors.append(gini_to_color(g if not np.isnan(g) else np.nan))
    col = PatchCollection(patches, facecolors=colors, edgecolors='white', linewidths=0.08, zorder=2)
    ax.add_collection(col)

print("Rendering map...")
fig = plt.figure(figsize=(18, 13))
fig.patch.set_facecolor('#F0EDE8')

ax_main = fig.add_axes([0.0, 0.05, 0.82, 0.90])
ax_main.set_facecolor('#C8DFF0')
draw_polys(ax_main, pen_polys, pen_gini)
ax_main.set_xlim(-9.5, 4.5)
ax_main.set_ylim(35.8, 44.0)
ax_main.set_aspect('equal')
ax_main.axis('off')

ax_can = fig.add_axes([0.01, 0.06, 0.22, 0.20])
ax_can.set_facecolor('#C8DFF0')
can_pts_all = []
for rings in can_polys:
    for ring in rings:
        can_pts_all.extend(ring)
if can_pts_all:
    can_arr = np.array(can_pts_all)
    draw_polys(ax_can, can_polys, can_gini)
    ax_can.set_xlim(can_arr[:,0].min()-0.1, can_arr[:,0].max()+0.1)
    ax_can.set_ylim(can_arr[:,1].min()-0.1, can_arr[:,1].max()+0.1)
    ax_can.set_aspect('equal')
    ax_can.axis('off')
    ax_can.set_title('Canarias', fontsize=9, pad=2)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.84, 0.15, 0.02, 0.55])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Gini Index', fontsize=13, labelpad=10)
cbar.ax.tick_params(labelsize=11)

nodata_patch = Patch(facecolor=(0.82,0.82,0.82,1), edgecolor='white', label='No data')
ax_main.legend(handles=[nodata_patch], loc='lower right', fontsize=10, framealpha=0.85)

fig.text(0.42, 0.97, 'Income Inequality across Spanish Municipalities',
         ha='center', fontsize=18, fontweight='bold')
fig.text(0.42, 0.93, 'Gini Index — INE Atlas de Distribución de Renta de los Hogares (2022)',
         ha='center', fontsize=13, color='#444')
fig.text(0.99, 0.01, 'Source: INE Atlas de Distribución de Renta de los Hogares (2022) & IGN',
         ha='right', fontsize=9, color='#666')

out = '/Users/finn/Desktop/TFG/output/spain_gini_map.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()
print(f"Map saved to {out}")

Reading shapefiles...
Rendering map...
Map saved to /Users/finn/Desktop/TFG/output/spain_gini_map.png
